# 📊 Evaluasi Sistem Rekomendasi Wisata Danau Toba
## Perbandingan Performa RAG vs CAG

**Model LLM:** Gemini 2.0 Flash (Google API)

### Metrik yang Diukur:
| Kategori | Metrik | Deskripsi |
|----------|--------|-----------|
| **Efficiency** | Response Time | Waktu total dari query hingga response |
| **Efficiency** | Cache Hit Rate (CHR) | Persentase query yang dilayani dari cache |
| **Retrieval** | RAG Recall | Keyword relevan yang ditemukan di retrieved docs |
| **Retrieval** | EIR | Effective Information Rate - info context yang digunakan |
| **Generation** | BERTScore F1 | Semantic similarity dengan ground truth |
| **Generation** | Completeness | Coverage keyword dalam response |
| **Generation** | Hallucination Rate | Informasi yang tidak ada di context |

## 1️⃣ Setup Environment
Install dependencies dan import libraries yang diperlukan.

In [ ]:
# Install Dependencies
!pip install -q python-dotenv sentence-transformers langchain langchain-community faiss-cpu bert-score pandas matplotlib seaborn

In [ ]:
# Setup & Imports
import sys, os, json, time
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from datetime import datetime
from bert_score import score as bert_score
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv

sys.path.append(os.path.abspath('../src'))
load_dotenv()
sns.set_style('whitegrid')
print("✅ Setup complete")

## 2️⃣ Load Model & Encoder
- **LLM:** Gemini 2.0 Flash via Google API (tidak memerlukan GPU lokal)
- **Encoder:** sentence-transformers/all-MiniLM-L12-v2 untuk embedding dokumen

In [ ]:
# Load Gemini 2.0 Flash & Encoder
from model import GeminiChatModel
gemini = GeminiChatModel(model_name="gemini-2.0-flash")
encoder = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L12-v2', model_kwargs={'device': 'cpu'})
print("✅ Gemini 2.0 Flash & Encoder loaded")

## 3️⃣ Build Knowledge Base
Load dokumen PDF → Split chunks → Buat FAISS vector database untuk retrieval.

In [ ]:
# Load Documents & Build Vector DB
tourism_dir = os.path.abspath('../data/tourism')
pdf_files = [os.path.join(tourism_dir, f) for f in os.listdir(tourism_dir) if f.endswith('.pdf')] if os.path.exists(tourism_dir) else []
pages = [p for pdf in pdf_files for p in PyPDFLoader(pdf).load()]
docs = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(pages) if pages else []
faiss_db = FAISS.from_documents(docs, encoder) if docs else None
print(f"✅ {len(pdf_files)} PDFs → {len(docs)} chunks")

## 4️⃣ Setup CAG System
Inisialisasi **Cache-Augmented Generation** untuk caching respons dan mempercepat query berulang.

In [ ]:
# Setup CAG System
try:
    from cag_system import CAGSystem
    cag = CAGSystem(gemini, encoder)
    if pdf_files: cag.load_documents(pdf_files, use_summaries=False)
    cag_ok = True
    print("✅ CAG ready")
except Exception as e:
    cag_ok = False
    print(f"⚠️ CAG not available: {e}")

## 5️⃣ Test Dataset
5 query uji tentang wisata Danau Toba dengan **ground truth** dan **keywords** untuk evaluasi.

In [ ]:
# Test Dataset
dataset = [
    {"q": "Rekomendasi wisata Danau Toba untuk keluarga?", "gt": "Pulau Samosir, Museum Huta Bolon, Pantai Parapat, Desa Tomok.", "kw": ["Samosir", "Parapat", "wisata", "Batak"]},
    {"q": "Biaya homestay di Danau Toba?", "gt": "Rp 150.000 - Rp 500.000 per malam.", "kw": ["homestay", "biaya", "Rp"]},
    {"q": "Makanan khas Batak?", "gt": "Saksang, Arsik, Naniura, Dali ni Horbo.", "kw": ["Saksang", "Arsik", "Batak"]},
    {"q": "Cara ke Pulau Samosir dari Parapat?", "gt": "Ferry setiap 30 menit, Rp 10.000.", "kw": ["ferry", "Parapat", "Samosir"]},
    {"q": "Waktu terbaik ke Danau Toba?", "gt": "Mei-September (musim kemarau).", "kw": ["waktu", "musim", "kemarau"]},
]
print(f"✅ {len(dataset)} queries")

## 6️⃣ Inference Functions
- **RAG:** Retrieve → Generate (tanpa cache)
- **CAG:** Check cache → Retrieve → Generate → Cache response

In [ ]:
# Inference Functions
def rag_infer(q, k=5):
    t0 = time.time()
    if not faiss_db: return {'resp': '', 'ctx': '', 'docs': [], 'time': 0, 'cache': False}
    d = faiss_db.similarity_search(q, k=k)
    ctx = "\n".join(x.page_content for x in d)
    resp = gemini.generate(f"Context: {ctx}\n\nPertanyaan: {q}\n\nJawaban:")
    return {'resp': resp, 'ctx': ctx, 'docs': d, 'time': time.time()-t0, 'cache': False}

def cag_infer(q, k=5):
    if not cag_ok: return {'resp': '', 'ctx': '', 'docs': [], 'time': 0, 'cache': False}
    t0 = time.time()
    r = cag.query(q, k=k)
    return {'resp': r.get('response',''), 'ctx': r.get('context',''), 'docs': r.get('retrieved_docs',[]), 'time': time.time()-t0, 'cache': r.get('cache_hit', False)}

print("✅ Inference ready")

## 7️⃣ Evaluation Metrics
| Metrik | Formula |
|--------|---------|
| **BERTScore F1** | Semantic similarity response vs ground truth |
| **Completeness** | % keywords yang muncul di response |
| **Hallucination** | % kalimat response tanpa grounding di context |
| **RAG Recall** | % keywords ditemukan di retrieved docs |
| **EIR** | % context words yang digunakan di response |

In [ ]:
# Metrics Functions
def bertscore_f1(p, r): P,R,F = bert_score(p, r, lang='id', verbose=False); return F.mean().item()
def completeness(resp, kw): return sum(k.lower() in resp.lower() for k in kw) / len(kw) if kw else 0
def hallucination(resp, ctx): 
    if not ctx: return 0
    sents = [s.strip() for s in resp.split('.') if len(s) > 20]
    return sum(not any(w in ctx.lower() for w in s.lower().split() if len(w)>4) for s in sents) / len(sents) if sents else 0
def rag_recall(docs, kw):
    if not docs or not kw: return 0
    ctx = " ".join(d.page_content.lower() if hasattr(d,'page_content') else str(d).lower() for d in docs)
    return sum(k.lower() in ctx for k in kw) / len(kw)
def eir(ctx, resp):
    if not ctx or not resp: return 0
    cw, rw = set(w.lower() for w in ctx.split() if len(w)>4), set(w.lower() for w in resp.split() if len(w)>4)
    return len(cw & rw) / len(cw) if cw else 0
print("✅ Metrics ready")

## 8️⃣ Run Evaluation
Eksekusi RAG dan CAG untuk setiap query, dengan rate limiting 2 detik per query.

In [ ]:
# 🚀 RUN EVALUATION
print("="*60 + "\n🚀 EVALUATING RAG vs CAG\n" + "="*60)
rag_res, cag_res = [], []

for i, d in enumerate(dataset, 1):
    print(f"\n[{i}/{len(dataset)}] {d['q'][:40]}...")
    
    r = rag_infer(d['q']); r.update({'gt': d['gt'], 'kw': d['kw']}); rag_res.append(r)
    print(f"  RAG: {r['time']:.2f}s")
    
    if cag_ok:
        c = cag_infer(d['q']); c.update({'gt': d['gt'], 'kw': d['kw']}); cag_res.append(c)
        print(f"  CAG: {c['time']:.2f}s {'📦' if c['cache'] else ''}")
    time.sleep(2)  # Rate limit

print(f"\n{'='*60}\n✅ Done! RAG:{len(rag_res)} CAG:{len(cag_res)}")

## 9️⃣ Calculate & Display Results
Hitung semua metrik dan tampilkan perbandingan RAG vs CAG dalam tabel.

In [ ]:
# 📊 CALCULATE METRICS
def calc_metrics(res):
    m = {'time':[], 'bert':[], 'comp':[], 'hall':[], 'recall':[], 'eir':[], 'cache':[]}
    for r in res:
        m['time'].append(r['time']); m['cache'].append(1 if r['cache'] else 0)
        m['bert'].append(bertscore_f1([r['resp']], [r['gt']]))
        m['comp'].append(completeness(r['resp'], r['kw']))
        m['hall'].append(hallucination(r['resp'], r['ctx']))
        m['recall'].append(rag_recall(r['docs'], r['kw']))
        m['eir'].append(eir(r['ctx'], r['resp']))
    return {k: np.mean(v) for k,v in m.items()}

rag_m = calc_metrics(rag_res)
cag_m = calc_metrics(cag_res) if cag_res else None
print("✅ Metrics calculated")

In [ ]:
# 📋 RESULTS TABLE
print("\n" + "="*60 + "\n📊 EVALUATION RESULTS\n" + "="*60)
metrics = ['Response Time (s)', 'Cache Hit Rate', 'BERTScore F1', 'Completeness', 'Hallucination', 'RAG Recall', 'EIR']
rag_v = [f"{rag_m['time']:.3f}", f"{rag_m['cache']*100:.0f}%", f"{rag_m['bert']:.4f}", f"{rag_m['comp']:.4f}", f"{rag_m['hall']:.4f}", f"{rag_m['recall']:.4f}", f"{rag_m['eir']:.4f}"]
data = {'Metric': metrics, 'RAG': rag_v}

if cag_m:
    data['CAG'] = [f"{cag_m['time']:.3f}", f"{cag_m['cache']*100:.0f}%", f"{cag_m['bert']:.4f}", f"{cag_m['comp']:.4f}", f"{cag_m['hall']:.4f}", f"{cag_m['recall']:.4f}", f"{cag_m['eir']:.4f}"]
    speedup = rag_m['time'] / cag_m['time'] if cag_m['time'] > 0 else 0

df = pd.DataFrame(data)
print(df.to_string(index=False))
if cag_m: print(f"\n🚀 CAG Speedup: {speedup:.2f}x")

## 🔟 Visualization
Bar charts untuk perbandingan visual: Response Time, BERTScore, dan Quality Metrics.

In [ ]:
# 📈 VISUALIZATION
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sys_names = ['RAG'] + (['CAG'] if cag_m else [])
colors = ['#3498db', '#e74c3c'][:len(sys_names)]

# Response Time
times = [rag_m['time']] + ([cag_m['time']] if cag_m else [])
ax[0].bar(sys_names, times, color=colors); ax[0].set_title('⚡ Response Time (s)')
for i,v in enumerate(times): ax[0].text(i, v, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

# BERTScore
berts = [rag_m['bert']] + ([cag_m['bert']] if cag_m else [])
ax[1].bar(sys_names, berts, color=colors); ax[1].set_title('🎯 BERTScore F1'); ax[1].set_ylim(0,1)
for i,v in enumerate(berts): ax[1].text(i, v, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')

# Quality Metrics
x = np.arange(4); w = 0.35
rv = [rag_m['comp'], rag_m['hall'], rag_m['recall'], rag_m['eir']]
ax[2].bar(x - w/2, rv, w, label='RAG', color='#3498db')
if cag_m:
    cv = [cag_m['comp'], cag_m['hall'], cag_m['recall'], cag_m['eir']]
    ax[2].bar(x + w/2, cv, w, label='CAG', color='#e74c3c')
ax[2].set_xticks(x); ax[2].set_xticklabels(['Comp', 'Hall', 'Recall', 'EIR'])
ax[2].set_title('📊 Quality'); ax[2].legend(); ax[2].set_ylim(0,1)

plt.tight_layout()
os.makedirs('../logs', exist_ok=True)
plt.savefig('../logs/eval_results.png', dpi=150)
plt.show()
print("✅ Saved: logs/eval_results.png")

## 📝 Summary & Export
Simpan hasil ke CSV & JSON, tampilkan key findings dari evaluasi.

In [ ]:
# 💾 SAVE & SUMMARY
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
df.to_csv(f'../logs/eval_{ts}.csv', index=False)
json.dump({'ts': ts, 'rag': rag_m, 'cag': cag_m}, open(f'../logs/eval_{ts}.json', 'w'), indent=2)

print("\n" + "="*60)
print("🎯 KEY FINDINGS")
print("="*60)
print(f"RAG: {rag_m['time']:.3f}s | BERTScore: {rag_m['bert']:.4f} | Completeness: {rag_m['comp']:.4f}")
if cag_m:
    print(f"CAG: {cag_m['time']:.3f}s | BERTScore: {cag_m['bert']:.4f} | Cache: {cag_m['cache']*100:.0f}%")
    print(f"\n💡 CAG is {speedup:.1f}x faster with comparable quality!")
print("\n✅ EVALUATION COMPLETE!")